In [18]:
import pandas as pd

# Load the dataset
df = pd.read_csv("Titanic-Dataset.csv")

# Basic exploration
print("Columns in dataset:\n", df.columns.tolist())
print("\nUnique Cabins:\n", df['Cabin'].unique())

print("\n--- Total Null Values per Column ---")
for i in df.columns:
    null_count = df[i].isnull().sum()
    if null_count > 0:
        print(f"{i} has {null_count} total null values")

# Uncomment to see unique values for each column
# for i in df:
#     print(f"Unique values in {i}: {df[i].unique()}\n")

FileNotFoundError: [Errno 2] No such file or directory: 'Titanic-Dataset.csv'

In [ ]:
# Extract titles from the Name column.
# Titles are useful because survival rate was discriminatory (saving children, rich, women, elders).
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

print("Unique Titles:\n", df['Title'].unique())
print("\nTitle Counts:\n", df['Title'].value_counts())

# Check the percentage of missing Age values for each Title group
print("\nPercentage of missing Age per Title:")
print(df.groupby('Title')['Age'].apply(lambda x: x.isnull().mean() * 100))

In [ ]:
# Impute missing Age values based on the mean age of the passenger's Title.
# This prevents introducing noise, as titles strongly correlate with age groups.
df['Age'] = df['Age'].fillna(df.groupby('Title')['Age'].transform('mean'))

print("Remaining null values in Age:", df['Age'].isnull().sum())

In [ ]:
# Cabins were often shared by families or people with the same ticket number.
# Create a dictionary mapping known Ticket numbers to their Cabin.
ticket_cabin_dict = df.dropna(subset=['Cabin']).set_index('Ticket')['Cabin'].to_dict()

# Fill missing Cabin values for passengers who share a Ticket number with someone whose cabin is known.
df['Cabin'] = df['Cabin'].fillna(df['Ticket'].map(ticket_cabin_dict))

print("Remaining null values in Cabin after Ticket mapping:", df['Cabin'].isnull().sum())
print("\nRemaining Null Cabins per Pclass:")
print(df.groupby('Pclass')['Cabin'].apply(lambda x: x.isnull().sum()))

In [ ]:
# 1. Pclass 2 and 3: Systematically ignored, so we fill remaining missing cabins with 'U' (Unknown)
df.loc[(df['Pclass'].isin([2, 3])) & (df['Cabin'].isnull()), 'Cabin'] = 'U'

# 2. Pclass 1: Assign cabin letters based on Fare intervals
# We use pd.qcut to divide the Pclass 1 fares into 3 equal-sized bins (quantiles)
pclass1_mask = df['Pclass'] == 1
df.loc[pclass1_mask, 'Fare_Bin'] = pd.qcut(df.loc[pclass1_mask, 'Fare'], q=3, labels=['Lower', 'Mid', 'Expensive'])

# Map the bins to the hierarchical cabin letters you identified
# Using 'B' for Most Expensive (A/B), 'C' for Mid (C/D), and 'E' for Lower
bin_to_cabin_map = {
    'Expensive': 'B',
    'Mid': 'C',
    'Lower': 'E'
}

# Apply the mapping only to Pclass 1 passengers who still have missing cabins
pclass1_missing_mask = (df['Pclass'] == 1) & (df['Cabin'].isnull())
df.loc[pclass1_missing_mask, 'Cabin'] = df.loc[pclass1_missing_mask, 'Fare_Bin'].map(bin_to_cabin_map)

# Drop the temporary 'Fare_Bin' column as we no longer need it
df = df.drop(columns=['Fare_Bin'])

print("Final remaining null values in Cabin:", df['Cabin'].isnull().sum())

In [ ]:
print("\n--- Total Null Values per Column ---")
for i in df.columns:
    null_count = df[i].isnull().sum()
    if null_count > 0:
        print(f"{i} has {null_count} total null values")


In [ ]:
df['Embarked'].unique()

In [ ]:
# embarked has very little missing data, so we just simply drop the instances with no data
df=df.dropna(subset=['Embarked'])

In [ ]:
print("\n--- Total Null Values per Column ---")
for i in df.columns:
    null_count = df[i].isnull().sum()
    if null_count > 0:
        print(f"{i} has {null_count} total null values")

In [ ]:
df.columns

In [ ]:
df["CabinCat"]=df['Cabin'].str[0]

In [ ]:
df.columns

In [ ]:
# dropping unecessary features
# Drop multiple columns by passing a list and using axis=1
# df = df.drop(['Cabin', 'Name'], axis=1)
# now further evaluating columns
# df['PassengerId'].unique()  # useless as it contains only numbering
# df=df.drop(["PassengerId"], axis=1)
df.columns

In [ ]:
# family size 
# sibsp represents the siblings abroad
# parch represents the parents abroad
# so we just put a family size column to add the familysize abroad
# df['familySize']=df['SibSp']+df['Parch']+1
# added plus 1 to include the person itself
# df=df.drop(['SibSp', 'Parch'], axis=1)
df.columns

In [ ]:
# Now we want to encode the categories into numerals

In [ ]:
# # The fixed apply() method:
# df['Gender'] = df['Gender'].apply(lambda x: 0 if x == 'male' else 1)
# # df['Gender'].value_counts()

In [2]:
# now the gender data has been corrupted
# fix
dff=pd.read_csv("Titanic-Dataset.csv")
df['Gender']=dff['Gender'].str.strip()
df['Gender'].value_counts()
# The fixed apply() method:
# df['Gender'] = df['Gender'].apply(lambda x: 0 if x == 'male' else 1)
# df['Gender'].value_counts()

FileNotFoundError: [Errno 2] No such file or directory: 'Titanic-Dataset.csv'

In [ ]:
# as fare is the collective fare so i need individual passenger fare
# 1. Count how many times each Ticket appears in the dataset
ticket_counts = df['Ticket'].value_counts()

# 2. Map those counts back to the dataframe to create a 'Group_Size' column
df['Group_Size'] = df['Ticket'].map(ticket_counts)

# 3. Calculate the true individual fare
df['Individual_Fare'] = df['Fare'] / df['Group_Size']

# 4. NOW we can finally drop the Ticket column (and the original misleading Fare)
df = df.drop(['Ticket', 'Fare'], axis=1)

In [3]:
df.columns

NameError: name 'df' is not defined

In [4]:
# now dealing with the embarked column
df['Embarked'].value_counts()

NameError: name 'df' is not defined

In [5]:
# encoding the categories of Embarked feature into numerals
df['Embarked']=df['Embarked'].str.strip()
df['Embarked'] = df['Embarked'].apply(lambda x: 0 if x == 'S' else 1 if x=='C' else 2)
df['Embarked'].value_counts()

NameError: name 'df' is not defined

In [6]:
df['CabinCat'].value_counts()

NameError: name 'df' is not defined

In [7]:
# Telling Pandas which text columns need to be split into 1s and 0s
columns_to_encode = ['CabinCat', 'Embarked', 'Title', 'Group_Size']
# Run the automatic one-hot encoding
df = pd.get_dummies(df, columns=columns_to_encode, drop_first=True, dtype=int)

NameError: name 'df' is not defined

In [8]:
df.columns

NameError: name 'df' is not defined

In [9]:
# df.drop("Gender_1")
columns_to_encode=['Gender']
df=pd.get_dummies(df, columns=columns_to_encode, drop_first=True, dtype=int)
df.columns

NameError: name 'df' is not defined

In [10]:
df['Gender'].value_counts()

NameError: name 'df' is not defined

In [11]:
columns_to_encode=['Pclass']
df=pd.get_dummies(df, columns=columns_to_encode, drop_first=True, dtype=int)
df.columns

NameError: name 'df' is not defined

In [12]:
columns_to_encode=['familySize']
df=pd.get_dummies(df, columns=columns_to_encode, drop_first=True, dtype=int)
df.columns

NameError: name 'df' is not defined

In [13]:
# This will drop ALL columns named 'Gender_1'
df = df.drop('Gender_1', axis=1)

NameError: name 'df' is not defined

In [14]:
df.columns

NameError: name 'df' is not defined

In [17]:
# now i am doing min/max normalization
from sklearn.preprocessing import MinMaxScaler

# 1. Initialize the scaler engine
scaler = MinMaxScaler()

# 2. Define ONLY the continuous columns
columns_to_scale = ['Age', 'Individual_Fare']

# 3. Fit and transform
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

NameError: name 'df' is not defined

In [ ]:
# Use .describe() to get the min, max, and statistical distribution
print(df[['Age', 'Individual_Fare']].describe())

In [ ]:
df.columns

In [ ]:
# Force every value in the column to be a strict integer
df['Survived'] = df['Survived'].astype(int)

# Verify that the only existing values are exactly 0 and 1
print("Unique values in Survived:", df['Survived'].unique())

# Verify the data type is an integer (int32 or int64)
print("Data type:", df['Survived'].dtype)

In [ ]:
df.columns

In [ ]:
from sklearn.model_selection import train_test_split
X=df.drop('Survived', axis=1)
y=df['Survived']

X_train, X_test, Y_train, Y_test=train_test_split(X,y, test_size=0.2, random_state=42)

# Check the sizes of our new datasets
print(f"Training data: {X_train.shape[0]} passengers")
print(f"Testing data: {X_test.shape[0]} passengers")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [16]:
# initialize and train model
model=LogisticRegression()
model.fit(X_train, Y_train)

NameError: name 'LogisticRegression' is not defined

In [ ]:
predictions=model.predict(X_test)
acc= accuracy_score(Y_test, predictions)
print(acc)